# PCA y clasificación de fallas APS en camiones Scania

**Inteligencia Artificial I — Entrega 1.** Los datos provienen del conjunto oficial *APS Failure at Scania Trucks*, UCI. No se crean registros sintéticos.

PCA es el método principal de reducción de dimensiones; un clasificador logístico realiza las predicciones sobre las componentes.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'entrenar.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.preprocessing import leer_scania, diagnostico_datos
import pandas as pd


## 1. Descargar y explorar datos reales
Ejecuta la descarga oficial antes de analizar. El archivo de entrenamiento contiene 60.000 registros y 170 variables numéricas anonimizadas.


In [ ]:
from scripts.descargar_datos import descargar
descargar()
datos = leer_scania(ROOT / 'data/raw/aps_failure_training_set.csv')
print(datos.shape)
display(datos.head())
display(datos.describe(include='all').T.head(12))


## 2. Diagnóstico real de calidad
Revisar valores faltantes, duplicados y balance de etiquetas antes de transformar las variables.


In [ ]:
reporte = diagnostico_datos(datos)
for k, v in reporte.items():
    print(k, ':', v)
display(datos.isna().mean().sort_values(ascending=False).head(15).to_frame('proporción_nulos'))


## 3. Entrenamiento y serialización
El código usa división estratificada 80/20 y un pipeline que aprende los parámetros de limpieza solo en entrenamiento, evita fuga de información y usa 20 componentes PCA.


In [ ]:
from entrenar import main
modelo, metadata = main()
print('Varianza explicada por PCA:', metadata['varianza_explica_pca'])
display(pd.DataFrame([metadata['metricas_validacion']]).T.rename(columns={0:'valor'}))


## 4. Cargar el modelo guardado
La siguiente prueba confirma que el archivo serializado se puede cargar y utilizar con un registro auténtico de UCI.


In [ ]:
import joblib
modelo_guardado = joblib.load(ROOT / 'models/modelo_pca_scania.joblib')
prueba = leer_scania(ROOT / 'data/raw/aps_failure_test_set.csv')
fila = prueba.drop(columns='class').iloc[[0]]
print('Predicción:', modelo_guardado.predict(fila)[0])
print('Etiqueta real:', prueba['class'].iloc[0])


## 5. Aplicación local
Desde la terminal ubicada en la raíz del proyecto ejecuta: `python -m streamlit run app/app.py`. Toma capturas auténticas de la consola, el modelo en `models/`, la app y el repositorio GitHub.
